# 2-1 タイタニック号のデータで集計と可視化を学ぶ / Learning Aggregation and Visualization with Titanic Data
『AIに頼んで動かす Python実務データ分析』第2章1節の参照用ノートブックです。 / Reference notebook for Chapter 2, Section 1.

`LANG` を `"ja"` または `"en"` にして、すべてのセルを上から実行します。データは seaborn に含まれているものを使うので、アップロードは不要です。
Set `LANG` and run all cells. The data comes with seaborn, so no upload is needed.

In [ ]:
LANG = "ja"   # "ja" / "en"
SAVE_FIGURES = True

In [ ]:
import os, subprocess, warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager as fm
JP_FONT = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
if LANG == "ja":
    if not os.path.exists(JP_FONT):
        subprocess.run("apt-get -y -qq install fonts-noto-cjk > /dev/null", shell=True, check=True)
    fm.fontManager.addfont(JP_FONT)
    plt.rcParams["font.family"] = fm.FontProperties(fname=JP_FONT).get_name()
plt.rcParams.update({"font.size": 8, "axes.edgecolor": "black", "axes.spines.top": False,
                     "axes.spines.right": False, "savefig.dpi": 300})
FIG_W = 4.5
def save(fig, name):
    fig.tight_layout()
    if SAVE_FIGURES:
        os.makedirs(f"figures/{LANG}", exist_ok=True)
        fig.savefig(f"figures/{LANG}/{name}.png", bbox_inches="tight")
    plt.show()

In [ ]:
L = {"ja": dict(rate="生存率（%）", sex={"female": "女性", "male": "男性"}, cls={1: "1等", 2: "2等", 3: "3等"},
                age="年齢", n="人"),
     "en": dict(rate="Survival rate (%)", sex={"female": "Female", "male": "Male"}, cls={1: "1st", 2: "2nd", 3: "3rd"},
                age="Age", n="")}[LANG]

## データの読み込みと確認 / Loading and checking the data

In [ ]:
import seaborn as sns
df = sns.load_dataset("titanic")
print(df.shape)
df.head()

In [ ]:
missing = pd.DataFrame({"missing": df.isna().sum(), "percent": (df.isna().mean() * 100).round(1)})
missing[missing.missing > 0]

## 生存率を比べる / Comparing survival rates (Figure 2-1-1)

In [ ]:
print("全体 / overall:", round(df.survived.mean() * 100, 1), "%", df.survived.value_counts().to_dict())
by_sex = df.groupby("sex").survived.mean() * 100
by_cls = df.groupby("pclass").survived.mean() * 100
fig, axs = plt.subplots(1, 2, figsize=(FIG_W, 2.2), sharey=True, gridspec_kw={"width_ratios": [2, 3]})
for ax, s, lab in [(axs[0], by_sex, L["sex"]), (axs[1], by_cls, L["cls"])]:
    ax.bar([lab[k] for k in s.index], s.values, color="#888888", edgecolor="black", lw=0.6, width=0.6)
    for i, v in enumerate(s.values): ax.text(i, v + 2, f"{v:.1f}", ha="center", fontsize=7)
axs[0].set_ylabel(L["rate"]); axs[0].set_ylim(0, 100)
save(fig, "fig2-1-1_survival_sex_class")
pd.DataFrame({"rate_%": by_sex.round(1)}), pd.DataFrame({"rate_%": by_cls.round(1)})

## 性別と客室クラスを組み合わせる / Combining sex and class (Figure 2-1-2)

In [ ]:
ct = pd.crosstab(df.sex, df.pclass, values=df.survived, aggfunc="mean") * 100
cnt = pd.crosstab(df.sex, df.pclass)
fig, ax = plt.subplots(figsize=(FIG_W * 0.7, 1.5))
im = ax.imshow(ct.values, cmap="Greys", vmin=0, vmax=100, aspect="auto")
nr, nc = im.get_array().shape
ax.set_xticks(np.arange(-0.5, nc, 1), minor=True); ax.set_yticks(np.arange(-0.5, nr, 1), minor=True)
ax.grid(which="minor", color="black", linewidth=0.8); ax.tick_params(which="minor", length=0)
for i in range(2):
    for j in range(3):
        v = ct.values[i, j]
        ax.text(j, i, f"{v:.1f}%\n({cnt.values[i, j]}{L['n']})", ha="center", va="center", fontsize=7,
                color="white" if v > 60 else "black")
ax.set_xticks(range(3), [L["cls"][c] for c in ct.columns]); ax.set_yticks(range(2), [L["sex"][s] for s in ct.index])
for sp in ax.spines.values(): sp.set_visible(False)
save(fig, "fig2-1-2_sex_class_heatmap")
ct.round(1)

## 年齢と生存率 / Age and survival (Figure 2-1-3)

In [ ]:
bins = [0, 12, 17, 29, 44, 59, 80]
labels = ["0–12", "13–17", "18–29", "30–44", "45–59", "60+"]
g = df.groupby(pd.cut(df.age, bins, labels=labels)).survived.agg(["mean", "size"])
fig, ax = plt.subplots(figsize=(FIG_W, 2.2))
ax.bar(labels, g["mean"] * 100, color="#888888", edgecolor="black", lw=0.6, width=0.6)
for i, (v, n) in enumerate(zip(g["mean"] * 100, g["size"])):
    ax.text(i, v + 2, f"{v:.1f}\n(n={n})", ha="center", fontsize=6.5)
ax.set_ylabel(L["rate"]); ax.set_xlabel(L["age"]); ax.set_ylim(0, 80)
save(fig, "fig2-1-3_age_survival")
g.assign(rate=(g["mean"] * 100).round(1))

## 欠損値が結果をゆがめる / Missing values can bias results

In [ ]:
print("年齢が分かる人の生存率 / with age:", round(df[df.age.notna()].survived.mean() * 100, 1), "%")
print("年齢が不明な人の生存率 / age missing:", round(df[df.age.isna()].survived.mean() * 100, 1), "%")
print("全体 / overall:", round(df.survived.mean() * 100, 1), "%")
(df.groupby("pclass").age.apply(lambda x: x.isna().mean()) * 100).round(1).rename("age_missing_%")